### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

In [14]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="Year the movie was released")
    rating:float=Field(description="Movie output out of 10")
    director:str=Field(description="The director of the movie")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about movie Bajirao Mastani")
response


Movie(title='Bajirao Mastani', year=2015, rating=8.2, director='Sanjay Leela Bhansali')

In [7]:
model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Provide details about movie Bajirao Mastani")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about movie Bajirao Mastani". We need to provide details. Could use the function to get movie details? The function signature: Movie takes director, rating, title, year. We need to fill those. We need to know details: director (Sanjay Leela Bhansali), rating maybe from some source? Could guess 8.2? But better to use function with rating maybe 8.2. Provide title "Bajirao Mastani", year 2015. Provide rating out of 10. Let\'s assume rating 8.2. We\'ll call function.', 'tool_calls': [{'id': 'fc_a34eb84d-8660-437e-9cff-938de595f753', 'function': {'arguments': '{"director":"Sanjay Leela Bhansali","rating":8.2,"title":"Bajirao Mastani","year":2015}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 186, 'prompt_tokens': 157, 'total_tokens': 343, 'completion_time': 0.391498376, 'completion_tokens_details': {'reasoning_tokens': 125}, 'prompt_time

### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160000000.0)

In [16]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=Movie
)
result = agent.invoke({
    "messages" : [HumanMessage("Provide details about the movie Inception")]
})
result

{'messages': [HumanMessage(content='Provide details about the movie Inception', additional_kwargs={}, response_metadata={}, id='ae8c08e2-8eda-4e44-912d-9ec49ffeb1d5'),
  AIMessage(content='{"title":"Inception","year":2010,"rating":8.8,"director":"Christopher Nolan"}', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about the movie Inception". The system expects output in the "Movie" JSON format as defined. Must include fields: title, year, rating, director. Must be compact JSON, valid, include all required fields.\n\nWe need to fill with correct details: Inception (2010), director Christopher Nolan, rating maybe 8.8 out of 10 (IMDb). Provide rating as number. Use maybe 8.8.\n\nThus JSON: {"title":"Inception","year":2010,"rating":8.8,"director":"Christopher Nolan"}\n\nMake sure compact (no spaces). Provide only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 235, 'total_tokens': 401, 'completion_time': 0.346624551, 'complet

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [12]:
from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [13]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Marion Cotillard', 'role': 'Mal'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [17]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.messages import HumanMessage

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages" : [HumanMessage("Extract contact info from: John Doe, john@example.com, (555) 123-4567")]
})
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='002bc3a5-25b1-4887-869c-19e17d1ea758'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user asks to extract contact info from a string. The system says respond with only the final JSON object, matching the ContactInfo schema: fields name, email, phone all required.\n\nWe need to output compact JSON. Provide name "John Doe", email "john@example.com", phone "(555) 123-4567". Ensure correct formatting.\n\nThus output: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 212, 'total_tokens': 345, 'completion_time': 0.279883207, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_time': 0.008988681, 'prompt_tokens_details': None, 'queue_